# 11 把 checkpoint 部署到 MuJoCo 闭环

        这一节把 ACT、SmolVLA 或 pi_0 checkpoint 放回 `SimpleEnv2`，让策略按 20 Hz 读取双相机图像和 6 维关节状态，并输出 7 维动作。评估会保存视频和 JSONL，同时报告旧几何成功与严格 `physical_success`。

        这是仿真闭环推理，不是把模型导出成服务。模型每次动作后都会重新读取环境观测；如果只在数据集上预测 GT 动作，那属于 open-loop replay，不能替代这里的成功率。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("请从 AMD ROCm 专题目录或 notebooks 子目录启动 Jupyter。")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"缺少素材：{path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


In [ ]:
import shlex

try:
    import yaml
except ImportError as exc:
    raise RuntimeError("当前环境缺少 PyYAML，请先执行 pip install pyyaml。") from exc


def require_project_layout():
    required = [
        PROJECT_ROOT / "train_model.py",
        PROJECT_ROOT / "env_config.py",
        PROJECT_ROOT / "asset" / "example_scene_y2.xml",
        PROJECT_ROOT / "mujoco_env" / "y_env2.py",
    ]
    missing = [path for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "PROJECT_ROOT 不是 04mujoco 教程目录，缺少：\n"
            + "\n".join(str(path) for path in missing)
        )
    return True


def dataset_report(dataset_root):
    dataset_root = Path(dataset_root)
    info_path = dataset_root / "meta" / "info.json"
    tasks_path = dataset_root / "meta" / "tasks.jsonl"
    if not info_path.exists():
        raise FileNotFoundError(f"找不到 LeRobot 元数据：{info_path}")
    info = json.loads(info_path.read_text(encoding="utf-8"))
    tasks = []
    if tasks_path.exists():
        tasks = [
            json.loads(line)["task"]
            for line in tasks_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
    features = info.get("features", {})
    rows = [
        ("repo_id", info.get("repo_id", "")),
        ("episodes", info.get("total_episodes", 0)),
        ("frames", info.get("total_frames", 0)),
        ("fps", info.get("fps", "")),
        ("state shape", features.get("observation.state", {}).get("shape")),
        ("action shape", features.get("action", {}).get("shape")),
        ("tasks", " / ".join(tasks)),
    ]
    md_table(["数据项", "读取结果"], rows)
    return info, tasks


def make_training_config(
    policy_type,
    dataset_repo_id,
    dataset_root,
    output_dir,
    steps,
    batch_size,
    chunk_size,
    n_action_steps,
    save_freq,
    seed=42,
):
    return {
        "dataset": {"repo_id": dataset_repo_id, "root": str(Path(dataset_root))},
        "policy": {
            "type": policy_type,
            "chunk_size": int(chunk_size),
            "n_action_steps": int(n_action_steps),
            "device": "cuda",
        },
        "save_checkpoint": True,
        "output_dir": str(Path(output_dir)),
        "batch_size": int(batch_size),
        "job_name": Path(output_dir).name,
        "resume": False,
        "seed": int(seed),
        "num_workers": 4,
        "steps": int(steps),
        "eval_freq": 0,
        "log_freq": max(1, min(50, int(steps))),
        "save_freq": int(save_freq),
        "use_policy_training_preset": True,
        "wandb": {
            "enable": False,
            "project": f"every_embodied_{policy_type}",
            "entity": None,
            "disable_artifact": True,
        },
    }


def write_training_config(name, config):
    config_dir = OUTPUT_ROOT / "configs"
    config_dir.mkdir(parents=True, exist_ok=True)
    path = config_dir / f"{name}.yaml"
    path.write_text(
        yaml.safe_dump(config, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )
    print("已写出配置：", path)
    return path


def training_command(config_path):
    return [
        sys.executable,
        str(PROJECT_ROOT / "train_model.py"),
        "--config_path",
        str(Path(config_path)),
    ]


def run_training(config_path, enabled=False):
    require_project_layout()
    command = training_command(config_path)
    print("$", shlex.join(command))
    if not enabled:
        print("当前只预览命令。确认配置后，把对应 RUN_* 开关改为 True。")
        return None
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{PROJECT_ROOT}:{env.get('PYTHONPATH', '')}"
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def show_rocm_resources():
    command = ["rocm-smi", "--showuse", "--showmemuse", "--showtemp"]
    print("$", shlex.join(command))
    if shutil.which(command[0]) is None:
        print("未找到 rocm-smi。确认当前机器是否安装 ROCm。")
        return
    subprocess.run(command, check=False)


In [ ]:
import random
import time

import numpy as np
from PIL import Image

try:
    import imageio.v2 as imageio
except ImportError:
    imageio = None


def find_pretrained_model(run_dir):
    run_dir = Path(run_dir)
    last = run_dir / "checkpoints" / "last" / "pretrained_model"
    if last.exists():
        return last
    candidates = sorted((run_dir / "checkpoints").glob("*/pretrained_model"))
    if not candidates:
        raise FileNotFoundError(f"没有在 {run_dir} 下找到 pretrained_model")
    return candidates[-1]


def image_tensor(array, size=(256, 256)):
    import torch

    image = Image.fromarray(array).convert("RGB").resize(size)
    value = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(value).permute(2, 0, 1).contiguous()


def load_policy(policy_type, policy_path, dataset_repo_id, dataset_root, device="cuda"):
    from lerobot.common.datasets.lerobot_dataset import LeRobotDatasetMetadata

    if policy_type == "act":
        from lerobot.common.policies.act.modeling_act import ACTPolicy as PolicyClass
    elif policy_type == "smolvla":
        from lerobot.common.policies.smolvla.modeling_smolvla import SmolVLAPolicy as PolicyClass
    elif policy_type == "pi0":
        from lerobot.common.policies.pi0.modeling_pi0 import PI0Policy as PolicyClass
    else:
        raise ValueError(f"不支持的 policy_type：{policy_type}")

    metadata = LeRobotDatasetMetadata(dataset_repo_id, root=dataset_root)
    policy = PolicyClass.from_pretrained(str(policy_path), dataset_stats=metadata.stats)
    policy.to(device)
    policy.eval()
    return policy


def strict_snapshot(env, initial_target_z, max_target_lift, max_lifted_run):
    target_pos = np.asarray(env.env.get_p_body(env.obj_target), dtype=np.float64)
    plate_pos = np.asarray(env.env.get_p_body("body_obj_plate_11"), dtype=np.float64)
    target_R = np.asarray(env.env.get_R_body(env.obj_target), dtype=np.float64)
    xy_dist = float(np.linalg.norm(target_pos[:2] - plate_pos[:2]))
    upright_cos = float(target_R[2, 2])
    gripper_open = bool(float(env.env.get_qpos_joint("rh_r1")[0]) < 0.1)
    tcp_high = bool(float(env.env.get_p_body("tcp_link")[2]) > 0.9)
    legacy_success = bool(env.check_success())
    physical_success = bool(
        legacy_success
        and max_target_lift >= 0.03
        and max_lifted_run >= 3
        and upright_cos >= 0.7
        and abs(float(target_pos[2] - plate_pos[2])) < 0.15
        and gripper_open
        and tcp_high
    )
    return {
        "legacy_success": legacy_success,
        "physical_success": physical_success,
        "xy_dist": xy_dist,
        "target_z": float(target_pos[2]),
        "plate_z": float(plate_pos[2]),
        "max_target_lift": float(max_target_lift),
        "max_lifted_run": int(max_lifted_run),
        "upright_cos": upright_cos,
        "gripper_open": gripper_open,
        "tcp_high": tcp_high,
    }


def run_closed_loop(
    policy,
    policy_type,
    instruction,
    seeds,
    output_dir,
    device="cuda",
    max_action_steps=300,
    render=True,
):
    from mujoco_env.y_env2 import SimpleEnv2
    import torch

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    xml_path = PROJECT_ROOT / "asset" / "example_scene_y2.xml"
    results = []

    for seed in seeds:
        np.random.seed(seed)
        random.seed(seed)
        env = SimpleEnv2(str(xml_path), action_type="joint_angle", state_type="joint_angle", seed=None)
        env.set_instruction(instruction)
        policy.reset()

        initial_target_z = float(env.env.get_p_body(env.obj_target)[2])
        max_target_lift = 0.0
        lifted_run = 0
        max_lifted_run = 0
        frames = []
        started = time.perf_counter()
        final = None

        action_step = 0
        while action_step < max_action_steps and env.env.is_viewer_alive():
            env.step_env()
            if not env.env.loop_every(HZ=20):
                continue

            state = env.get_joint_state()[:6]
            agent_image, wrist_image = env.grab_image()
            observation = {
                "observation.state": torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0),
                "observation.image": image_tensor(agent_image).to(device).unsqueeze(0),
                "observation.wrist_image": image_tensor(wrist_image).to(device).unsqueeze(0),
                "task": [instruction],
            }
            with torch.inference_mode():
                action = policy.select_action(observation)[0, :7].detach().cpu().numpy()
            action[6] = np.clip(action[6], 0.0, 1.0)
            env.step(action.astype(np.float32))

            target_z = float(env.env.get_p_body(env.obj_target)[2])
            lift = target_z - initial_target_z
            max_target_lift = max(max_target_lift, lift)
            lifted_run = lifted_run + 1 if lift >= 0.03 else 0
            max_lifted_run = max(max_lifted_run, lifted_run)
            final = strict_snapshot(env, initial_target_z, max_target_lift, max_lifted_run)

            if action_step % 2 == 0:
                frames.append(np.asarray(agent_image))
            if render:
                env.render()
            action_step += 1
            if final["physical_success"]:
                break

        video_path = output_dir / f"{policy_type}_seed{seed}.mp4"
        video_saved = False
        if frames and imageio is not None:
            try:
                imageio.mimsave(video_path, frames, fps=10, quality=8)
                video_saved = True
            except Exception as exc:
                print("视频保存失败：", repr(exc))
        elif frames:
            print("未安装 imageio，跳过视频保存。可执行 pip install imageio imageio-ffmpeg 后重跑。")
        row = {
            "policy_type": policy_type,
            "seed": int(seed),
            "instruction": instruction,
            "action_steps": int(action_step),
            "elapsed_s": round(time.perf_counter() - started, 3),
            "video": str(video_path) if video_saved else None,
            **(final or {}),
        }
        print(json.dumps(row, ensure_ascii=False))
        results.append(row)
        env.env.close_viewer()

    result_path = output_dir / "results.jsonl"
    result_path.write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in results),
        encoding="utf-8",
    )
    physical = sum(row.get("physical_success", False) for row in results)
    print(f"physical_success = {physical}/{len(results)}")
    print("结果文件：", result_path)
    return results


## Checkpoint 1：选择模型、数据统计和评估 seed


In [ ]:
POLICY_TYPE = os.environ.get("POLICY_TYPE", "act")  # act / smolvla / pi0
DATASET_REPO_ID = os.environ.get("DATASET_REPO_ID", "datawhale_eai_pnp_language")
EVAL_DATA_ROOT = Path(
    os.environ.get("EVAL_DATA_ROOT", DATA_ROOT / "omy_pnp_language")
).expanduser()
MODEL_RUN_DIR = Path(
    os.environ.get("MODEL_RUN_DIR", MODEL_ROOT / f"{POLICY_TYPE}_rocm_full")
).expanduser()
TASK_TEXT = os.environ.get("TASK_TEXT", "Place the blue mug on the plate.")
EVAL_SEEDS = [int(value) for value in os.environ.get("EVAL_SEEDS", "1000,1001,1002,1003").split(",")]
MAX_ACTION_STEPS = int(os.environ.get("MAX_ACTION_STEPS", "300"))
RUN_CLOSED_LOOP = False

print("policy type =", POLICY_TYPE)
print("model run =", MODEL_RUN_DIR)
print("dataset =", EVAL_DATA_ROOT)
print("task =", TASK_TEXT)
print("seeds =", EVAL_SEEDS)


先用 4 个固定 seed 做 smoke，但 4 条不具有统计代表性。模型通过小面板后，再扩到至少 20–30 个 held-out seeds，并把红杯、蓝杯分别统计。评估时不要读取 target/plate 坐标、数据集 phase 或 GT 动作作为策略输入。


## Checkpoint 2：检查数据与 checkpoint


In [ ]:
require_project_layout()
if (EVAL_DATA_ROOT / "meta" / "info.json").exists():
    dataset_report(EVAL_DATA_ROOT)
else:
    print("缺少评估数据统计：", EVAL_DATA_ROOT)

try:
    POLICY_PATH = Path(os.environ.get("POLICY_PATH", ""))
    if not str(POLICY_PATH) or str(POLICY_PATH) == ".":
        POLICY_PATH = find_pretrained_model(MODEL_RUN_DIR)
    print("policy path =", POLICY_PATH)
except FileNotFoundError as exc:
    POLICY_PATH = None
    print(exc)


## Checkpoint 3：加载策略并执行闭环


In [ ]:
if RUN_CLOSED_LOOP:
    if POLICY_PATH is None:
        raise FileNotFoundError("请先完成训练，或通过 POLICY_PATH 指定 pretrained_model。")
    if POLICY_TYPE == "pi0" and not os.environ.get("HF_TOKEN"):
        print("本地 checkpoint 通常可离线加载；若仍需读取 gated 配置，请先私密设置 HF_TOKEN。")
    show_rocm_resources()
    policy = load_policy(
        policy_type=POLICY_TYPE,
        policy_path=POLICY_PATH,
        dataset_repo_id=DATASET_REPO_ID,
        dataset_root=EVAL_DATA_ROOT,
        device="cuda",
    )
    results = run_closed_loop(
        policy=policy,
        policy_type=POLICY_TYPE,
        instruction=TASK_TEXT,
        seeds=EVAL_SEEDS,
        output_dir=OUTPUT_ROOT / f"{POLICY_TYPE}_closed_loop",
        device="cuda",
        max_action_steps=MAX_ACTION_STEPS,
        render=True,
    )
else:
    print("闭环评估默认关闭。确认 DISPLAY、checkpoint、数据统计和 seed 后，将 RUN_CLOSED_LOOP 改为 True。")


## Checkpoint 4：读取成功率和失败类型


In [ ]:
result_path = OUTPUT_ROOT / f"{POLICY_TYPE}_closed_loop" / "results.jsonl"
if result_path.exists():
    rows = [json.loads(line) for line in result_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    physical = sum(row.get("physical_success", False) for row in rows)
    legacy = sum(row.get("legacy_success", False) for row in rows)
    md_table(
        ["口径", "结果"],
        [
            ("legacy_success", f"{legacy}/{len(rows)}"),
            ("physical_success", f"{physical}/{len(rows)}"),
        ],
    )
    for row in rows:
        if not row.get("physical_success"):
            print(
                "失败 seed",
                row["seed"],
                "xy=", round(row.get("xy_dist", float("nan")), 4),
                "lift=", round(row.get("max_target_lift", float("nan")), 4),
                "upright=", round(row.get("upright_cos", float("nan")), 4),
            )
else:
    print("尚未生成闭环结果：", result_path)


如果 `legacy_success > physical_success`，优先查看失败视频，通常是推杯、未真正夹住、运输中掉落或只满足终态几何条件。下一步回到 02–06 的诊断 Notebook，把失败定位到 approach、contact、lift、transport、release 中的具体阶段。
